In [3]:
!pip install streamlit folium streamlit-folium plotly pandas
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 3s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼npm notice
npm notice New major version of npm available! 10.8.2 -> 11.12.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.12.1
npm notice To update run: npm install -g npm@11.12.1
npm notice
⠴

In [12]:
import os
if os.path.exists("app.py"):
  os.remove("app.py")

In [25]:
%%writefile app.py
import streamlit as st
import pandas as pd
import folium
from streamlit_folium import st_folium
import plotly.graph_objects as go
from datetime import datetime

# --- DAY 4: LAYOUT FINALIZATION & PAGE CONFIG ---
st.set_page_config(page_title="AI Aircraft Health & Efficiency Dashboard", layout="wide")

# --- DAY 1, 3 & 5: REFINED DATA INTEGRATION (API STRUCTURE) ---
API_RESPONSE = {
    "live_telemetry": {
        "tail": "N123AF",
        "risk": "High",
        "health": 62,
        "lat": 34.0522,
        "lon": -118.2437,
        "origin": "LAX (Los Angeles)",
        "destination": "JFK (New York)",
        "fuel_eff": "88%"
    },
    # Day 3: Real explainability_shap list provided by API
    "explainability_shap": [
        {"feature": "Baseline", "value": 100},
        {"feature": "Turbine Temp", "value": -12.5},
        {"feature": "Vibration Index", "value": -15.0},
        {"feature": "Oil Pressure", "value": 5.0},
        {"feature": "Flight Cycles", "value": -10.5},
        {"feature": "Weather Stress", "value": -5.0}
    ],
    "flight_history": [
        {"Date": "2024-01-10", "Flight_ID": "FL-991", "Origin": "DXB", "Destination": "LHR", "Status": "Completed"},
        {"Date": "2024-01-11", "Flight_ID": "FL-992", "Origin": "LHR", "Destination": "SIN", "Status": "Completed"},
        {"Date": "2024-01-12", "Flight_ID": "FL-993", "Origin": "SIN", "Destination": "LAX", "Status": "Completed"},
        # Day 5 Validation: Live telemetry matches this final entry
        {"Date": "2024-01-13", "Flight_ID": "FL-994", "Origin": "LAX (Los Angeles)", "Destination": "JFK (New York)", "Status": "In-Flight"}
    ]
}

# --- IMPLEMENTING NEW NAVIGATION FEATURES ---
st.sidebar.title("✈️ FYP Navigation")
st.sidebar.markdown("---")
page = st.sidebar.radio("Select View:", ["Live Dashboard", "Mission Logs"])

# Shortcuts to data
live = API_RESPONSE["live_telemetry"]
history_df = pd.DataFrame(API_RESPONSE["flight_history"])
shap_list = API_RESPONSE["explainability_shap"]

# ==========================================
# PAGE 1: LIVE DASHBOARD
# ==========================================
if page == "Live Dashboard":
    st.title("🛰️ AI Live Telemetry Analytics")
    st.caption(f"Last Updated: {datetime.now().strftime('%H:%M:%S')}")

    # Day 4: Top Metrics Row (Responsive Layout)
    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Tail Number", live['tail'])
    m2.metric("Risk Assessment", live['risk'], delta="In-Flight", delta_color="inverse")
    m3.metric("Health Score", f"{live['health']}/100")
    m4.metric("Fuel Efficiency", live['fuel_eff'])

    st.divider()

    # Main Visuals Row
    col_map, col_diag = st.columns([1.5, 1])

    with col_map:
        st.subheader("📍 Interactive Mission Map")
        # Day 2: Live-to-History Data Binding
        # Markers show Origin and Destination pulled from telemetry
        m = folium.Map(location=[37.0902, -95.7129], zoom_start=4, tiles="CartoDB positron")

        marker_popup = f"""
        <b>Tail:</b> {live['tail']}<br>
        <b>Origin:</b> {live['origin']}<br>
        <b>Destination:</b> {live['destination']}
        """

        folium.Marker(
            [live['lat'], live['lon']],
            popup=marker_popup,
            tooltip="View Flight Path Data",
            icon=folium.Icon(color="red", icon="plane")
        ).add_to(m)

        st_folium(m, width='stretch', height=500)

    with col_diag:
        st.subheader("📊 Diagnostic Explainability")

        # Health Gauge
        gauge = go.Figure(go.Indicator(
            mode = "gauge+number", value = live['health'],
            gauge = {
                'axis': {'range': [0, 100]},
                'bar': {'color': "#3498db"},
                'steps': [
                    {'range': [0, 50], 'color': "red"},
                    {'range': [50, 80], 'color': "orange"},
                    {'range': [80, 100], 'color': "green"}]
            }
        ))
        gauge.update_layout(height=240, margin=dict(l=20, r=20, t=30, b=10))
        st.plotly_chart(gauge, use_container_width=True)

        # Day 3: Waterfall Chart (Reading real explainability_shap list)
        df_shap = pd.DataFrame(shap_list)
        waterfall = go.Figure(go.Waterfall(
            orientation="v",
            measure=["absolute"] + ["relative"] * (len(df_shap)-1),
            x=df_shap['feature'],
            y=df_shap['value'],
            decreasing={"marker": {"color": "#e74c3c"}},
            increasing={"marker": {"color": "#2ecc71"}},
            totals={"marker": {"color": "#3498db"}}
        ))
        waterfall.update_layout(height=280, title="SHAP Value Analysis")
        st.plotly_chart(waterfall, use_container_width=True)

    # Day 5: Full System Data Validation
    last_hist = history_df.iloc[-1]
    if live['origin'] == last_hist['Origin'] and live['destination'] == last_hist['Destination']:
        st.success(f"✅ Data Validation Passed: Live Telemetry matches Flight History ID: {last_hist['Flight_ID']}")

# ==========================================
# PAGE 2: MISSION LOGS
# ==========================================
elif page == "Mission Logs":
    st.title("📋 Fleet Mission Logs")
    st.write("Historical data records for maintenance and efficiency audits.")

    # Day 1: Display Table of Past Flights
    st.dataframe(history_df, use_container_width=True)

    # Day 4: Download Report Button
    st.markdown("---")
    csv_data = history_df.to_csv(index=False).encode('utf-8')
    st.download_button(
        label="📥 Download Current Flight History (CSV)",
        data=csv_data,
        file_name=f"mission_logs_{live['tail']}.csv",
        mime='text/csv',
    )

    # Step 7: Validation Logic in Logs
    st.info(f"Integrity Check: Final history log entry matches current live telemetry state for {live['tail']}.")

Overwriting app.py


In [26]:
!pip install pyngrok
from pyngrok import ngrok

# Authenticate
!ngrok config add-authtoken 35c1sNj52z3eAfvg7Pdj1hV1xiT_7JQnM1NZ9WXZvCiE6YuME

# Kill any existing tunnels
ngrok.kill()

# Open a tunnel on port 8501 (Streamlit's port)
public_url = ngrok.connect(8501)
print("Click this link to see your dashboard:", public_url)

# Run the app
!streamlit run app.py

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Click this link to see your dashboard: NgrokTunnel: "https://847a-34-19-115-86.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.19.115.86:8501

2026-03-30 21:43:14.628 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-30 21:43:14.637 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-30 21:43:19.126 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_cont